![Logo 1](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech1.jpg)
<div class="alert alert-block alert-info">
<h1> Komputerowe wspomaganie tłumaczenia </h1>
<h2> 6,7. <i>Preprocessing i postprocessing</i> [laboratoria]</h2> 
<h3>Rafał Jaworski (2021)</h3>
</div>

![Logo 2](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech2.jpg)

Na dzisiejszych zajęciach zajmiemy się niezwykle przydatnymi narzędziami wspomagającymi pracę tłumacza. W odróżnieniu od dotychczas poznanych, nie są one oparte na pamięci tłumaczeń, ani na słownikach. Chodzi o techniki preprocessingu i postprocessingu.

Proces tłumaczenia przeprowadzony w pełni profesjonalnie składa się z wielu faz, które angażują nie tylko tłumaczy, ale także kierowników projektu, analityków, czy korektorów. Każda z tych osób do swojej pracy może wykorzystywać system informatyczny, do którego na początku całego procesu trafiają pliki do tłumaczenia. Oznacza to, że zanim tekst źródłowy trafi do tłumacza, system ma jeszcze szansę coś w nim zmienić. A kiedy tłumacz wykona już swoją pracę, można uruchomić kolejny mechanizm, który zmodyfikuje tłumaczenie przed oddaniem go do zamawiającego. Jak się domyślamy, modyfikacje tekstu przed przekazaniem go do tłumacza nazywamy **preprocessingiem**, natomiast te dokonywane po wykonaniu tłumaczenia (ale przed zwróceniem go do klienta) nazywamy **postprocessingiem**. Terminy te, będące mało zgrabnymi kalkami z języka angielskiego, mają wersje prawdziwie polskie: przetwarzanie wstępne i końcowe. Wersje te są jednak stosowane na tyle rzadko, że mogą jedynie wprowadzić zamieszanie (co w gruncie rzeczy jest dość smutne).

Typowe operacje w fazie preprocessingu obejmują:
* identyfikację tagów xmlowych (które często są później wyświetlane w interfejsie CAT-a jako jeden niepodzielny znak)
* identyfikację segmentów, których nie należy tłumaczyć (na przykład składających się z samych liczb)

* identyfikację dat i jednostek miary w tekście źródłowym

We wszystkich tych operacjach niezwykle przydatne okazują się wyrażenia regularne.

### Ćwiczenie 1: Używając wyrażeń regularnych napisz funkcję do znajdowania wszystkich tagów XML w tekście. Funkcja powinna zwracać pozycje, na których znalazła tagi.

In [1]:
import re

def find_tags(text):
    pattern = r"<[^>]+>"
    results = []

    for match in re.finditer(pattern, text):
        results.append((match.group(), match.start()))

    return results

### Ćwiczenie 2: Używając wyrażeń regularnych napisz funkcję do identyfikacji segmentów, których nie należy tłumaczyć. Zastosuj wymyślone przez siebie kryteria. Funkcja is_translatable powinna zwracać True, jeśli segment powinien być przetłumaczony przez tłumacza (zwykłe zdanie). False powinno być zwrócone, kiedy segment jest nieprzetłumaczalny i powinien zostać skopiowany (np. 4.2.1.)

In [2]:
def is_translatable(segment):
    if re.fullmatch(r"\s*", segment):
        return False

    if re.fullmatch(r"[\d\.\-_/()]+", segment):
        return False

    if re.fullmatch(r"\d+(\.\d+)*", segment):
        return False

    if re.fullmatch(r"[A-Z]{2,}", segment):
        return False

    if re.search(r"https?://|www\.", segment):
        return False

    if re.fullmatch(r"[A-Za-z0-9_\-]+@[A-Za-z0-9_\-]+\.[A-Za-z]{2,}", segment):
        return False

    if re.fullmatch(r"\d{1,2}[:\-]\d{2}", segment):
        return False

    if re.fullmatch(r"[^\w\s]+", segment):
        return False

    return True

### Ćwiczenie 3: Używając wyrażeń regularnych napisz funkcję do identyfikacji i interpretacji 5 wybranych przez siebie formatów daty. Funkcja powinna zwracać pozycje, na których odnalazła daty oraz dzień, miesiąc i rok, które ta data reprezentuje.

In [3]:
def find_dates(text):
    patterns = [
        r"\b(\d{2})\.(\d{2})\.(\d{4})\b",
        r"\b(\d{4})-(\d{2})-(\d{2})\b",
        r"\b(\d{2})/(\d{2})/(\d{4})\b",
        r"\b(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{4})\b",
        r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{1,2}),\s+(\d{4})\b"
    ]

    month_map = {
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
        "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
        "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
    }

    results = []

    for pattern in patterns:
        for match in re.finditer(pattern, text):
            start = match.start()

            groups = match.groups()

            if pattern == patterns[0]:
                day, month, year = groups
            elif pattern == patterns[1]:
                year, month, day = groups
            elif pattern == patterns[2]:
                day, month, year = groups
            elif pattern == patterns[3]:
                day, month_str, year = groups
                month = month_map[month_str]
            else:
                month_str, day, year = groups
                month = month_map[month_str]

            results.append((start, int(day), int(month), int(year)))

    return sorted(results, key=lambda x: x[0])

Po preprocessingu i tłumaczeniu czas na postprocessing. Ponieważ wykonywany jest on na przetłumaczonym tekście, jego głównym zadaniem jest eliminacja błędów popełnionych przez tłumacza w fazie tłumaczenia. Podczas postprocessingu najczęściej wykonuje się:
* korektę pisowni dla języka docelowego
* usuwanie błędów typograficznych z tekstu (np. wielokrotne spacje, brak spacji po przecinku)
Stanowi to bardzo ważne wsparcie dla edytorów i korektorów, czyli osób sprawdzających pracę tłumacza.

Jednak nowoczesne CAT-y potrafią coś jeszcze. Są w stanie w sprytny sposób wykorzystać kombinację pre- i postprocessingu do wyręczenia tłumacza w żmudnych i technicznych czynnościach. Wykonajmy następujące ćwiczenie:

### Ćwiczenie 4: Wykorzystując funkcję find_dates napisz funkcję do obsługi dat w tłumaczeniu. Wejściem jest segment źródłowy oraz docelowy, które zawierają daty, przy czym daty te mogą być w różnych formatach. Dodatkowym parametrem wejściowym jest nazwa oczekiwanego formatu daty w tłumaczeniu (np. "Europe", "US", "digit-dot". Funkcja najpierw sprawdza, czy liczba dat w tłumaczeniu zgadza się z liczbą dat w segmencie źródłowym oraz czy odpowiadające sobie daty wskazują na ten sam dzień. Jeśli nie, wypisywane jest stosowne ostrzeżenie. Oczekiwanym wyjściem jest segment docelowy, w którym wszystkie daty są w żądanym formacie. 

In [4]:
def format_date(day, month, year, date_format):
    if date_format == "Europe":
        return f"{day:02d}.{month:02d}.{year}"
    if date_format == "US":
        return f"{month:02d}/{day:02d}/{year}"
    if date_format == "digit-dot":
        return f"{year}-{month:02d}-{day:02d}"
    return f"{day:02d}.{month:02d}.{year}"


def correct_dates(source_segment, target_segment, date_format):
    source_dates = find_dates(source_segment)
    target_dates = find_dates(target_segment)

    if len(source_dates) != len(target_dates):
        print("Warning: different number of dates in source and target")

    min_len = min(len(source_dates), len(target_dates))

    for i in range(min_len):
        s = source_dates[i][1:]
        t = target_dates[i][1:]
        if s != t:
            print("Warning: mismatched dates at position", i)

    result = target_segment

    for start, d, m, y in reversed(target_dates):
        new_date = format_date(d, m, y, date_format)
        result = result[:start] + new_date + result[start:]

    return result

Co jeszcze można zrobić? Zajmijmy się tagami XML. Z punktu widzenia tłumacza najlepiej byłoby, gdyby mógł przetłumaczyć segment źródłowy zawierający tagi XML na język docelowy zupełnie ignorując te tagi. Ponieważ jednak tagi muszą jakoś znaleźć się w segmencie docelowym, przydałaby się jakaś "magiczna różdżka", która przeniosłaby wszystkie tagi ze źródła do tłumaczenia na mniej więcej dobre miejsca. Spełnijmy marzenie tłumacza!

Rozważmy następujący algorytm: na wejściu mamy segment źródłowy zawierający tagi oraz segment docelowy bez tagów. Dokonujemy tokenizacji segmentu źródłowego tak, aby tagi były osobnymi tokenami. Następnie przeprowadźmy tokenizację segmentu docelowego. Gdy to jest gotowe, możemy zabrać się za przenoszenie (kopiowanie) tagów z segmentu źródłowego do docelowego.

![Transfer tagów](../lab/img/tagtransfer.png)

Gdzie w segmencie docelowym powinien trafić tag? Przede wszystkim pomiędzy tokeny - nie chcemy rozbijać słów tagami. Pytanie tylko, pomiędzy które tokeny? Jeśli sytuacja jest taka, jak powyżej, kiedy segment źródłowy i docelowy mają taką samą liczbę słów nie będących tagami, przenosimy tagi na odpowiadające pozycje w tłumaczeniu. Natomiast jeśli długość tłumaczenia jest inna niż źródła, należy obliczać te pozycje w sposób proporcjonalny - jeśli np. mamy tag w źródle na pozycji 3, a tłumaczenie jest dwa razy dłuższe niż źródło, tag powinien być przeniesiony do tłumaczenia na pozycję 6. W przypadku niecałkowitych wartości proporcji stosujemy zaokrąglenia.

### Ćwiczenie 5: Zaimplementuj opisany algorytm transferu tagów.

In [5]:
import re

def tokenize(text):
    tokens = []
    spans = []
    for m in re.finditer(r"<[^>]+>|\S+", text):
        tokens.append(m.group())
        spans.append((m.start(), m.end()))
    return tokens, spans


def transfer_tags(source_segment, target_segment):
    source_tokens, _ = tokenize(source_segment)
    target_tokens, _ = tokenize(target_segment)

    source_tags = [(i, t) for i, t in enumerate(source_tokens) if re.fullmatch(r"<[^>]+>", t)]
    target_non_tags = [i for i, t in enumerate(target_tokens) if not re.fullmatch(r"<[^>]+>", t)]

    if len(source_tags) == 0 or len(target_non_tags) == 0:
        return target_segment

    tag_positions = []

    for idx, _ in source_tags:
        ratio = idx / max(len(source_tokens) - 1, 1)
        target_idx = int(round(ratio * (len(target_tokens) - 1)))
        target_idx = min(target_idx, len(target_tokens) - 1)
        tag_positions.append(target_idx)

    tag_positions = sorted(set(tag_positions))

    result_tokens = target_tokens[:]

    for pos, (_, tag) in zip(tag_positions, source_tags):
        while pos < len(result_tokens) and re.fullmatch(r"<[^>]+>", result_tokens[pos]):
            pos += 1
        result_tokens.insert(pos, tag)

    return " ".join(result_tokens)